# WiSARv1 Dataset Organization Investigation

This notebook performs a **read-only structural investigation** of a WiSARv1 dataset. It inventories paths, directory names, filenames, and metadata/manifests without opening image pixel data, modifying the dataset, copying files, extracting archives, resizing images, or training models.

The outputs are small aggregate text/CSV reports intended to answer whether defensible flight, recording-session, sequence, scene, or collection identifiers exist for leakage-safe splitting. Findings are labeled `documented`, `observed`, or `unknown`; different names are never treated as proof of different flights.

In [11]:
from __future__ import annotations

import csv
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

# Set WISAR_DATASET_ROOT or change this value after mounting Google Drive in Colab.
DATASET_ROOT = Path(os.environ.get("WISAR_DATASET_ROOT", "/content/drive/MyDrive/WiSARv1"))
if not DATASET_ROOT.exists() and Path("data/raw/WiSARD").is_dir():
    DATASET_ROOT = Path("data/raw/WiSARD")
REPORT_DIR = Path("results/dataset_audit")
TREE_MAX_DEPTH = 4
MAX_METADATA_BYTES = 2_000_000
MAX_TEXT_LINES_PER_FILE = 2_000

METADATA_EXTENSIONS = {
    ".csv", ".json", ".xml", ".yaml", ".yml", ".txt", ".tsv", ".mat",
    ".ini", ".cfg", ".conf", ".toml", ".md", ".md5", ".log",
}
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
    ".gif", ".ppm", ".pgm", ".dng", ".heic",
}
SEARCH_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "timestamp", "gps", "trajectory", "video", "mission",
)
GROUPING_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "mission", "run", "take", "trip", "set",
)


In [4]:
def tokenize_name(value: str) -> list[str]:
    """Split names into stable lowercase alphanumeric tokens without opening files."""
    return [token for token in re.split(r"[^a-zA-Z0-9]+", value.lower()) if token]


In [12]:
if not DATASET_ROOT.exists() or not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Set DATASET_ROOT to the mounted WiSARv1 directory; not found: {DATASET_ROOT}"
    )

root_resolved = DATASET_ROOT.resolve()
file_records = []
directory_records = []
metadata_records = []
extension_counts = Counter()
extension_by_directory = Counter()
directory_file_counts = Counter()
name_tokens = Counter()
directory_name_tokens = Counter()
tree_lines = [f"{root_resolved.name}/"]

# os.scandir reads directory entries and stat information; it does not decode image pixels.
stack = [(root_resolved, 0)]
while stack:
    current, depth = stack.pop()
    try:
        entries = sorted(os.scandir(current), key=lambda entry: (not entry.is_dir(follow_symlinks=False), entry.name.lower()))
    except (OSError, PermissionError) as error:
        directory_records.append({"relative_directory": str(current.relative_to(root_resolved)), "status": f"unreadable: {error}"})
        continue

    relative_current = current.relative_to(root_resolved)
    directory_records.append({"relative_directory": "." if relative_current == Path(".") else str(relative_current), "status": "read"})
    if depth <= TREE_MAX_DEPTH:
        tree_lines.extend([f"{'  ' * (depth + 1)}{'[D] ' if entry.is_dir(follow_symlinks=False) else '[F] '}{entry.name}" for entry in entries])

    for entry in entries:
        entry_path = Path(entry.path)
        relative_path = entry_path.relative_to(root_resolved)
        if entry.is_dir(follow_symlinks=False):
            directory_name_tokens.update(tokenize_name(entry.name))
            stack.append((entry_path, depth + 1))
            continue
        if not entry.is_file(follow_symlinks=False):
            continue

        suffix = entry_path.suffix.lower() or "[no_extension]"
        relative_directory = str(relative_path.parent)
        extension_counts[suffix] += 1
        extension_by_directory[(relative_directory, suffix)] += 1
        directory_file_counts[relative_directory] += 1
        tokens = tokenize_name(entry.name)
        name_tokens.update(tokens)
        record = {
            "relative_path": str(relative_path),
            "relative_directory": relative_directory,
            "filename": entry.name,
            "extension": suffix,
            "size_bytes": entry.stat(follow_symlinks=False).st_size,
            "name_tokens": ",".join(tokens),
        }
        file_records.append(record)
        if suffix in METADATA_EXTENSIONS:
            metadata_records.append(record.copy())

print(f"Dataset root: {root_resolved}")
print(f"Directories observed: {len(directory_records):,}")
print(f"Files observed: {len(file_records):,}")
print(f"Metadata-like files: {len(metadata_records):,}")
print("No image file was opened as pixel data.")

Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Directories observed: 1
Files observed: 1
Metadata-like files: 0
No image file was opened as pixel data.


In [13]:
metadata_matches = []
metadata_snippets = []
pattern_counts = Counter()
pattern_examples = defaultdict(list)

for record in metadata_records:
    path = root_resolved / record["relative_path"]
    filename_lower = record["filename"].lower()
    filename_terms = [term for term in SEARCH_TERMS if term in filename_lower]
    content_terms = []
    content = ""
    content_status = "not_read"
    if record["extension"] != ".mat" and record["size_bytes"] <= MAX_METADATA_BYTES:
        try:
            content = path.read_text(encoding="utf-8", errors="replace")
            content_status = "read"
        except (OSError, UnicodeError) as error:
            content_status = f"unreadable: {error}"
    elif record["extension"] == ".mat":
        content_status = "binary_mat_not_decoded"
    else:
        content_status = "skipped_over_size_limit"

    if content:
        content_lower = content.lower()
        content_terms = [term for term in SEARCH_TERMS if term in content_lower]
        for line_number, line in enumerate(content.splitlines()[:MAX_TEXT_LINES_PER_FILE], start=1):
            line_terms = [term for term in SEARCH_TERMS if term in line.lower()]
            if line_terms and len(metadata_snippets) < 200:
                metadata_snippets.append({
                    "relative_path": record["relative_path"],
                    "line_number": line_number,
                    "terms": ",".join(line_terms),
                    "snippet": re.sub(r"\s+", " ", line.strip())[:240],
                })

    all_terms = sorted(set(filename_terms + content_terms))
    if all_terms:
        evidence_status = "documented" if content_terms else "observed"
        metadata_matches.append({
            "relative_path": record["relative_path"],
            "extension": record["extension"],
            "filename_terms": ",".join(filename_terms),
            "content_terms": ",".join(content_terms),
            "terms": ",".join(all_terms),
            "content_status": content_status,
            "evidence_status": evidence_status,
        })

for record in file_records:
    source_name = f"{record['relative_directory']}/{record['filename']}"
    for pattern, label in (
        (r"(?:^|[^a-zA-Z])(?:flight|sequence|session|recording|collection|scene|mission|run|take|trip|set)[-_]?[a-zA-Z0-9]+", "grouping_term_with_value"),
        (r"(?:^|[^a-zA-Z])\d{2,}(?:[^a-zA-Z]|$)", "numeric_identifier"),
        (r"[A-Za-z]+[_-]\d+", "label_number"),
    ):
        matches = re.findall(pattern, source_name, flags=re.IGNORECASE)
        if matches:
            pattern_counts[label] += len(matches)
            for match in matches[:3]:
                if len(pattern_examples[label]) < 10:
                    pattern_examples[label].append(match.strip(" _-"))

candidate_rows = []
for record in file_records:
    path_parts = Path(record["relative_path"]).parts
    for part in path_parts:
        part_lower = part.lower()
        matched_terms = [term for term in GROUPING_TERMS if term in part_lower]
        has_identifier_shape = bool(re.search(r"(?:^|[_-])(?:\d+|[a-z]+\d+)(?:$|[_-])", part_lower))
        if matched_terms or has_identifier_shape:
            candidate_rows.append({
                "candidate": part,
                "source_path": record["relative_path"],
                "matched_terms": ",".join(matched_terms),
                "evidence_status": "observed",
                "interpretation": "Naming/path pattern only; not proof of an independent flight or session.",
            })

# Canonicalize RGB/thermal paths only for comparison; this does not alter source paths.
sensor_groups = defaultdict(lambda: {"rgb": set(), "thermal": set()})
for record in file_records:
    parts = list(Path(record["relative_path"]).parts)
    sensors = {part.lower() for part in parts if part.lower() in {"rgb", "thermal", "visible", "infrared", "ir"}}
    if not sensors:
        continue
    sensor = "rgb" if "rgb" in sensors or "visible" in sensors else "thermal"
    canonical_parts = [part.lower() for part in parts if part.lower() not in {"rgb", "thermal", "visible", "infrared", "ir"}]
    canonical = "/".join(canonical_parts[:-1]) if canonical_parts else "."
    sensor_groups[canonical][sensor].add(record["filename"].lower())

sensor_rows = []
for canonical, groups in sorted(sensor_groups.items()):
    has_both = bool(groups["rgb"] and groups["thermal"])
    sensor_rows.append({
        "canonical_path_without_sensor": canonical,
        "rgb_file_count": len(groups["rgb"]),
        "thermal_file_count": len(groups["thermal"]),
        "shared_filename_count": len(groups["rgb"] & groups["thermal"]),
        "evidence_status": "observed" if has_both else "unknown",
        "interpretation": (
            "RGB and thermal occur under a shared canonical path; verify timestamps/metadata before grouping."
            if has_both else
            "No paired path observed here; this does not prove different flights or sessions."
        ),
    })

print(f"Metadata files matching investigation terms: {len(metadata_matches):,}")
print(f"Candidate grouping path/name observations: {len(candidate_rows):,}")
print(f"RGB/thermal canonical groups: {len(sensor_rows):,}")

Metadata files matching investigation terms: 0
Candidate grouping path/name observations: 0
RGB/thermal canonical groups: 0


In [16]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def write_csv(filename: str, rows: list[dict], fieldnames: list[str]) -> None:
    output_path = REPORT_DIR / filename
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

extension_rows = [
    {"extension": extension, "file_count": count}
    for extension, count in sorted(extension_counts.items(), key=lambda item: (-item[1], item[0]))
]
directory_extension_rows = [
    {"relative_directory": directory, "extension": extension, "file_count": count}
    for (directory, extension), count in sorted(extension_by_directory.items())
]
directory_rows = [
    {"relative_directory": directory, "file_count": count}
    for directory, count in sorted(directory_file_counts.items())
]
pattern_rows = [
    {"pattern_type": label, "match_count": pattern_counts[label], "examples": "; ".join(pattern_examples[label])}
    for label in sorted(pattern_counts)
]

write_csv("extension_counts.csv", extension_rows, ["extension", "file_count"])
write_csv("directory_extension_counts.csv", directory_extension_rows, ["relative_directory", "extension", "file_count"])
write_csv("directory_file_counts.csv", directory_rows, ["relative_directory", "file_count"])
write_csv("metadata_term_matches.csv", metadata_matches, ["relative_path", "extension", "filename_terms", "content_terms", "terms", "content_status", "evidence_status"])
write_csv("metadata_term_snippets.csv", metadata_snippets, ["relative_path", "line_number", "terms", "snippet"])
write_csv("naming_patterns.csv", pattern_rows, ["pattern_type", "match_count", "examples"])
write_csv("grouping_candidates.csv", candidate_rows[:500], ["candidate", "source_path", "matched_terms", "evidence_status", "interpretation"])
write_csv("rgb_thermal_grouping.csv", sensor_rows[:500], ["canonical_path_without_sensor", "rgb_file_count", "thermal_file_count", "shared_filename_count", "evidence_status", "interpretation"])

(REPORT_DIR / "directory_tree.txt").write_text("\n".join(tree_lines) + "\n", encoding="utf-8")

reported_documented = sum(row["evidence_status"] == "documented" for row in metadata_matches)
reported_observed = sum(row["evidence_status"] == "observed" for row in metadata_matches) + len(candidate_rows)
summary = f"""WiSARv1 organization investigation
=================================
Dataset root: {root_resolved}
Files observed: {len(file_records):,}
Directories observed: {len(directory_records):,}
Metadata-like files observed: {len(metadata_records):,}
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: {reported_documented:,} metadata files contain search terms
observed: {reported_observed:,} filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports
-------
{chr(10).join(sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()))}
"""
(REPORT_DIR / "organization_investigation_report.txt").write_text(summary, encoding="utf-8")

print(summary)


WiSARv1 organization investigation
Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Files observed: 1
Directories observed: 1
Metadata-like files observed: 0
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: 0 metadata files contain search terms
observed: 0 filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports
